# Modern Seasons
The Premier League seasons from the most recent, 2024-2025, back to 2017-2018, have a more rich format than 1992-1993 through 2016-2017. Presumably, this is due to the inclusion of the Opta-supplied expected goals and expected assists data.

In any case, the parsing of the HTML will have to be in (at least) two separate branches because of this.

In [1]:
from pathlib import Path

import pandas as pd
import polars as pl

In [ ]:
for f in sorted(
    p
    for p in Path("data/premier-league").glob("*.html")
    if p.is_file() and p.stat().st_size > 0
):
    try:
        df = pd.read_html(
            f,
            flavor="lxml",
            dtype_backend="pyarrow",
            encoding="utf-8",
        )[0]
    except:
        n_cols = None
    else:
        n_cols = df.shape[1]
    finally:
        print(f"{f.name}: {n_cols}")
else:
    del df

In [3]:
THIRTY_SEVEN_COLUMN_INDEX: tuple[tuple[str, str], ...] = (
    ("Demographics", "Rk"),
    ("Demographics", "Player"),
    ("Demographics", "Nation"),
    ("Demographics", "Pos"),
    ("Demographics", "Squad"),
    ("Demographics", "Age"),
    ("Demographics", "Born"),
    ("Playing Time", "MP"),
    ("Playing Time", "Starts"),
    ("Playing Time", "Min"),
    ("Playing Time", "90s"),
    ("Performance", "Gls"),
    ("Performance", "Ast"),
    ("Performance", "G+A"),
    ("Performance", "G-PK"),
    ("Performance", "PK"),
    ("Performance", "PKatt"),
    ("Performance", "CrdY"),
    ("Performance", "CrdR"),
    ("Expected", "xG"),
    ("Expected", "npxG"),
    ("Expected", "xAG"),
    ("Expected", "npxG+xAG"),
    ("Progression", "PrgC"),
    ("Progression", "PrgP"),
    ("Progression", "PrgR"),
    ("Per 90 Minutes", "Gls"),
    ("Per 90 Minutes", "Ast"),
    ("Per 90 Minutes", "G+A"),
    ("Per 90 Minutes", "G-PK"),
    ("Per 90 Minutes", "G+A-PK"),
    ("Per 90 Minutes", "xG"),
    ("Per 90 Minutes", "xAG"),
    ("Per 90 Minutes", "xG+xAG"),
    ("Per 90 Minutes", "npxG"),
    ("Per 90 Minutes", "npxG+xAG"),
    ("Matches", "hyperlink"),
)

TWENTY_FIVE_COLUMN_INDEX: tuple[tuple[str, str], ...] = (
    ("Demographics", "Rk"),
    ("Demographics", "Player"),
    ("Demographics", "Nation"),
    ("Demographics", "Pos"),
    ("Demographics", "Squad"),
    ("Demographics", "Age"),
    ("Demographics", "Born"),
    ("Playing Time", "MP"),
    ("Playing Time", "Starts"),
    ("Playing Time", "Min"),
    ("Playing Time", "90s"),
    ("Performance", "Gls"),
    ("Performance", "Ast"),
    ("Performance", "G+A"),
    ("Performance", "G-PK"),
    ("Performance", "PK"),
    ("Performance", "PKatt"),
    ("Performance", "CrdY"),
    ("Performance", "CrdR"),
    ("Per 90 Minutes", "Gls"),
    ("Per 90 Minutes", "Ast"),
    ("Per 90 Minutes", "G+A"),
    ("Per 90 Minutes", "G-PK"),
    ("Per 90 Minutes", "G+A-PK"),
    ("Matches", "hyperlink"),
)

CHAMPIONS = {
    "2024-2025": "Liverpool",
    "2023-2024": "Manchester City",
    "2022-2023": "Manchester City",
    "2021-2022": "Manchester City",
    "2020-2021": "Manchester City",
    "2019-2020": "Liverpool",
    "2018-2019": "Manchester City",
    "2017-2018": "Manchester City",
    "2016-2017": "Chelsea",
    "2015-2016": "Leicester City",
    "2014-2015": "Chelsea",
    "2013-2014": "Manchester City",
    "2012-2013": "Manchester Utd",
    "2011-2012": "Manchester City",
    "2010-2011": "Manchester Utd",
    "2009-2010": "Chelsea",
    "2008-2009": "Manchester Utd",
    "2007-2008": "Manchester Utd",
    "2006-2007": "Manchester Utd",
    "2005-2006": "Chelsea",
    "2004-2005": "Chelsea",
    "2003-2004": "Arsenal",
    "2002-2003": "Manchester Utd",
    "2001-2002": "Arsenal",
    "2000-2001": "Manchester Utd",
    "1999-2000": "Manchester Utd",
    "1998-1999": "Manchester Utd",
    "1997-1998": "Arsenal",
    "1996-1997": "Manchester Utd",
    "1995-1996": "Manchester Utd",
    "1994-1995": "Blackburn",
    "1993-1994": "Manchester Utd",
    "1992-1993": "Manchester Utd",
}

In [4]:
full_df = pd.read_html(
    Path("data/premier-league/fbref_2024-2025.html"),
    flavor="lxml",
    dtype_backend="pyarrow",
    encoding="utf-8",
)[0]

full_df.columns = THIRTY_SEVEN_COLUMN_INDEX
full_df.head()

Demographics                                                            \
            Rk             Player   Nation    Pos        Squad Age  Born   
0            1         Max Aarons  eng ENG     DF  Bournemouth  24  2000   
1            2  Joshua Acheampong  eng ENG     DF      Chelsea  18  2006   
2            3        Tyler Adams   us USA     MF  Bournemouth  25  1999   
3            4   Tosin Adarabioyo  eng ENG     DF      Chelsea  26  1997   
4            5      Simon Adingra   ci CIV  FW,MF     Brighton  22  2002   

  Playing Time               ... Per 90 Minutes                           \
            MP Starts   Min  ...            Ast   G+A  G-PK G+A-PK    xG   
0            3      1    86  ...            0.0   0.0   0.0    0.0   0.0   
1            4      2   170  ...            0.0   0.0   0.0    0.0  0.12   
2           28     21  1965  ...           0.14  0.14   0.0   0.14  0.07   
3           22     15  1409  ...           0.06  0.13  0.06   0.13  0.06   
4           29     12  1097  ...           0.16  0.33  0.16   0.33   0.2   

                                Matches  
    xAG xG+xAG  npxG npxG+xAG hyperlink  
0   0.0    0.0   0.0      0.0   Matches  
1   0.0   0.12  0.12     0.12   Matches  
2  0.05   0.12  0.07     0.12   Matches  
3  0.01   0.07  0.06     0.07   Matches  
4   0.2    0.4   0.2      0.4   Matches  

[5 rows x 37 columns]

In [5]:
small_df = pd.read_html(
    Path("data/premier-league/fbref_2015-2016.html"),
    flavor="lxml",
    dtype_backend="pyarrow",
    encoding="utf-8",
)[0]

small_df.columns = TWENTY_FIVE_COLUMN_INDEX
small_df.head()

Demographics                                                             \
            Rk               Player   Nation Pos          Squad Age  Born   
0            1  Patrick van Aanholt   nl NED  DF     Sunderland  24  1990   
1            2       Rolando Aarons  eng ENG  MF  Newcastle Utd  19  1995   
2            3           Almen Abdi   ch SUI  MF        Watford  28  1986   
3            4        Tammy Abraham  eng ENG  FW        Chelsea  17  1997   
4            5         Charlie Adam  sct SCO  MF     Stoke City  29  1985   

  Playing Time               ... Performance                 Per 90 Minutes  \
            MP Starts   Min  ...          PK PKatt CrdY CrdR            Gls   
0           33     33  2970  ...           0     0    2    0           0.12   
1           10      3   381  ...           0     0    1    0           0.24   
2           32     25  1974  ...           0     0    4    0           0.09   
3            2      0    55  ...           0     0    0    0           0.00   
4           22     12  1044  ...           0     0    6    1           0.09   

                             Matches  
    Ast   G+A  G-PK G+A-PK hyperlink  
0  0.09  0.21  0.12   0.21   Matches  
1  0.24  0.47  0.24   0.47   Matches  
2  0.00  0.09  0.09   0.09   Matches  
3  0.00  0.00  0.00   0.00   Matches  
4  0.09  0.17  0.09   0.17   Matches  

[5 rows x 25 columns]

In [6]:
small_df.columns

MultiIndex([(  'Demographics',        'Rk'),
            (  'Demographics',    'Player'),
            (  'Demographics',    'Nation'),
            (  'Demographics',       'Pos'),
            (  'Demographics',     'Squad'),
            (  'Demographics',       'Age'),
            (  'Demographics',      'Born'),
            (  'Playing Time',        'MP'),
            (  'Playing Time',    'Starts'),
            (  'Playing Time',       'Min'),
            (  'Playing Time',       '90s'),
            (   'Performance',       'Gls'),
            (   'Performance',       'Ast'),
            (   'Performance',       'G+A'),
            (   'Performance',      'G-PK'),
            (   'Performance',        'PK'),
            (   'Performance',     'PKatt'),
            (   'Performance',      'CrdY'),
            (   'Performance',      'CrdR'),
            ('Per 90 Minutes',       'Gls'),
            ('Per 90 Minutes',       'Ast'),
            ('Per 90 Minutes',       'G+A'),
          

The `Rk` (rank) column is not helpful here, as it is just the default range index + 1, so we drop it. Also the -1th column is a Web hyperlink artifact and is useless. Without the rest of the birth date, the `Born` column doesn't seem to help anything because `Age` is included. All of the `Per 90 Minutes` columns can go, because they should be able to be re-calculated.

Desired Columns:
  - `Player`
  - `Squad`
  - `Age`
  - `Min` (minutes played)
  - `Gls` (goals)
  - `Ast` (assists)

In [7]:
small_df.loc[
    :, pd.IndexSlice[:, ["Squad", "Player", "Age", "Min", "Gls", "Ast"]]
][("Performance", "Gls")].value_counts(dropna=False)

(Performance, Gls)
0      295
1       89
2       55
3       28
5       23
Gls     22
4       21
6       11
8        7
11       6
7        6
9        5
10       3
24       2
12       2
13       2
15       2
16       1
25       1
18       1
17       1
Name: count, dtype: int64[pyarrow]

What is going on in the `Gls` column? It has 22 rows with value `"Gls"`, not a number?

In [8]:
valid = small_df.loc[
    (small_df.loc[:, ("Demographics", "Age")].str.isnumeric())
    & (small_df[("Performance", "Gls")].str.isnumeric())
    & (small_df[("Performance", "Ast")].str.isnumeric()),
    :
]
valid.loc[:, ("Performance", "Gls")] = valid.loc[:, ("Performance", "Gls")].astype("int16[pyarrow]")
valid.loc[:, ("Performance", "Ast")] = valid.loc[:, ("Performance", "Ast")].astype("int16[pyarrow]")

valid.loc[:, ("Performance", "Gls")].value_counts(dropna=False)

(Performance, Gls)
0     295
1      89
2      55
3      28
5      23
4      21
6      11
8       7
11      6
7       6
9       5
10      3
24      2
12      2
13      2
15      2
16      1
25      1
18      1
17      1
Name: count, dtype: int64[pyarrow]

In [9]:
valid.loc[
    :,
    pd.IndexSlice[
        ("Demographics", "Playing Time", "Performance"), ["Age", "Min", "Gls", "Ast"]
    ],
].astype("int16[pyarrow]")

Demographics Playing Time Performance    
             Age          Min         Gls Ast
0             24         2970           4   3
1             19          381           1   1
2             28         1974           2   0
3             17           55           0   0
4             29         1044           1   1
..           ...          ...         ...  ..
578           30         1065           1   2
579           30         1062           0   1
580           22         2488           2   1
581           28          788           3   0
582           20         1918           1   1

[561 rows x 4 columns]

In [10]:
desired = small_df.loc[
    :,
    pd.IndexSlice[
        ("Demographics", "Playing Time", "Performance"),
        ["Squad", "Player", "Age", "Min", "Gls", "Ast"],
    ],
]
desired

Demographics                          Playing Time Performance    
               Squad               Player Age          Min         Gls Ast
0         Sunderland  Patrick van Aanholt  24         2970           4   3
1      Newcastle Utd       Rolando Aarons  19          381           1   1
2            Watford           Almen Abdi  28         1974           2   0
3            Chelsea        Tammy Abraham  17           55           0   0
4         Stoke City         Charlie Adam  29         1044           1   1
..               ...                  ...  ..          ...         ...  ..
578   Manchester Utd         Ashley Young  30         1065           1   2
579  Manchester City       Pablo Zabaleta  30         1062           0   1
580   Crystal Palace        Wilfried Zaha  22         2488           2   1
581         West Ham         Mauro Zárate  28          788           3   0
582          Chelsea           Kurt Zouma  20         1918           1   1

[583 rows x 6 columns]

In [69]:
print(desired.columns.droplevel(0))
desired.columns = desired.columns.droplevel(0)
desired.head()

Index(['Squad', 'Player', 'Age', 'Min', 'Gls', 'Ast'], dtype='object')


,Squad,Player,Age,Min,Gls,Ast
0,Sunderland,Patrick van Aanholt,24,2970,4,3
1,Newcastle Utd,Rolando Aarons,19,381,1,1
2,Watford,Almen Abdi,28,1974,2,0
3,Chelsea,Tammy Abraham,17,55,0,0
4,Stoke City,Charlie Adam,29,1044,1,1


In [11]:
def html_to_flat_dataframe(path_to_html: Path) -> pd.DataFrame:
    """The HTML will bring in a MultiIndex column; single-level index is desired."""
    html_read = pd.read_html(
        path_to_html, flavor="lxml", dtype_backend="pyarrow", encoding="utf-8"
    )
    df: pd.DataFrame = html_read[0]
    if df.shape[1] == 37:
        df.columns = THIRTY_SEVEN_COLUMN_INDEX
    elif df.shape[1] == 25:
        df.columns = TWENTY_FIVE_COLUMN_INDEX
    else:
        _msg: str = (
            f"Unexpected number of columns, {len(df)}, "
            f"from file '{path_to_html.resolve()}'."
        )
        raise RuntimeError(_msg)

    desired_columns: list[str, str, str, str, str, str] = [
        "Squad",
        "Player",
        "Age",
        "Min",
        "Gls",
        "Ast",
    ]
    # If Age, Gls, or Ast value is a non-number string,
    # it's an HTML parsing artifact and is to be dropped
    if df.shape[1] == 25:
        subset: pd.DataFrame = df.loc[
            (df.loc[:, ("Demographics", "Age")].str.isnumeric())
            & (df[("Performance", "Gls")].str.isnumeric())
            & (df[("Performance", "Ast")].str.isnumeric()),
            pd.IndexSlice[
                ("Demographics", "Playing Time", "Performance"),
                desired_columns,
            ],
        ].copy()
    else:
        subset: pd.DataFrame = df.loc[
            :,
            pd.IndexSlice[
                ("Demographics", "Playing Time", "Performance"),
                desired_columns,
            ],
        ].copy()
    subset.columns = subset.columns.droplevel(0)
    subset["Age"] = subset["Age"].astype("int16[pyarrow]")
    subset["Min"] = subset["Min"].astype("int16[pyarrow]")
    subset["Gls"] = subset["Gls"].astype("int16[pyarrow]")
    subset["Ast"] = subset["Ast"].astype("int16[pyarrow]")

    return subset

In [11]:
df16 = html_to_flat_dataframe(Path("data/premier-league/fbref_2015-2016.html"))

In [12]:
df16.dtypes

Squad     string[pyarrow]
Player    string[pyarrow]
Age        int16[pyarrow]
Min        int16[pyarrow]
Gls        int16[pyarrow]
Ast        int16[pyarrow]
dtype: object

## Season 2024-2025
Champions Liverpool

In [17]:
df25 = html_to_flat_dataframe(
    Path("data/premier-league/fbref_2024-2025.html")
).sort_values(by="Gls", ascending=False)

In [18]:
# pl.from_pandas(df25).sort(pl.col("Gls"), descending=True)
subset25 = pl.from_pandas(df25).filter(pl.col("Squad") == CHAMPIONS["2024-2025"])
subset25.with_columns(
    pl.col("Gls").truediv(pl.col("Gls").sum()).alias("prop_total_goals")
)


Squad,Player,Age,Min,Gls,Ast,prop_total_goals
str,str,i16,i16,i16,i16,f64
"""Liverpool""","""Mohamed Salah""",32,3371,29,18,0.341176
"""Liverpool""","""Luis Díaz""",27,2399,13,5,0.152941
"""Liverpool""","""Cody Gakpo""",25,1935,10,4,0.117647
"""Liverpool""","""Diogo Jota""",27,1196,6,3,0.070588
"""Liverpool""","""Dominik Szoboszlai""",23,2491,6,6,0.070588
…,…,…,…,…,…,…
"""Liverpool""","""Vitezslav Jaros""",23,12,0,0,0.0
"""Liverpool""","""Caoimhín Kelleher""",25,900,0,0,0.0
"""Liverpool""","""Jarell Quansah""",21,495,0,0,0.0


## Season 2023-2024

In [15]:
df24 = pl.from_pandas(
    html_to_flat_dataframe(Path("data/premier-league/fbref_2023-2024.html"))
)
df24.filter(
    pl.col("Squad").eq(CHAMPIONS["2023-2024"])
).with_columns(
    pl.col("Gls").truediv(pl.col("Gls").sum()).alias("prop_total_goals").cast(pl.Float32)
).sort(by="Gls", descending=True)

Squad,Player,Age,Min,Gls,Ast,prop_total_goals
str,str,i16,i16,i16,i16,f32
"""Manchester City""","""Erling Haaland""",23,2552,27,5,0.287234
"""Manchester City""","""Phil Foden""",23,2857,19,8,0.202128
"""Manchester City""","""Julián Álvarez""",23,2647,11,8,0.117021
"""Manchester City""","""Rodri""",27,2931,8,9,0.085106
"""Manchester City""","""Bernardo Silva""",28,2578,6,9,0.06383
…,…,…,…,…,…,…
"""Manchester City""","""Matheus Nunes""",24,661,0,2,0.0
"""Manchester City""","""Stefan Ortega""",30,635,0,0,0.0
"""Manchester City""","""Cole Palmer""",21,11,0,0,0.0


# Season 2022-2023

In [163]:
df23 = pl.from_pandas(
    html_to_flat_dataframe(Path("data/premier-league/fbref_2022-2023.html"))
)
df23.filter(
    pl.col("Squad").eq(CHAMPIONS["2022-2023"])
).with_columns(
    pl.col("Gls").truediv(pl.col("Gls").sum()).alias("prop_total_goals").cast(pl.Float32)
).sort(by="Gls", descending=True).slice(0, 1)

Squad,Player,Age,Min,Gls,Ast,prop_total_goals
str,str,i16,i16,i16,i16,f32
"""Manchester City""","""Erling Haaland""",22,2769,36,8,0.391304


# All Seasons, Iteratively
Let's do a `for` loop, but then make it prettier at some point

In [12]:
all_seasons: list[pl.DataFrame, ...] = [None] * len(CHAMPIONS)
for i, (season, champ) in enumerate(CHAMPIONS.items()):
    pldf = pl.from_pandas(
        html_to_flat_dataframe(Path(f"data/premier-league/fbref_{season}.html")),
    )
    max_prop_goals_for_champ: pl.DataFrame = (
        pldf.filter(pl.col("Squad").eq(champ))
        .with_columns(
            pl.lit(season).alias("Season"),
            pl.lit(champ).alias("Champion"),
            pl.col("Gls").sum().alias("total_goals").cast(pl.Int16),
            pl.col("Gls")
            .truediv(pl.col("Gls").sum())
            .alias("prop_total_goals")
            .cast(pl.Float32),
        )
        .sort(by="prop_total_goals", descending=True)
        .slice(0, 1)
    )
    all_seasons[i] = max_prop_goals_for_champ
    # print(f"SEASON {season}:\n\t{max_prop_goals_for_champ}")

In [13]:
print(
    pl.concat(all_seasons)
    .select(
        "Season",
        "Champion",
        "Player",
        "Gls",
        "total_goals",
        "prop_total_goals",
    )
    .sort(by="prop_total_goals", descending=True)
)

shape: (33, 6)
┌───────────┬─────────────────┬───────────────────┬─────┬─────────────┬──────────────────┐
│ Season    ┆ Champion        ┆ Player            ┆ Gls ┆ total_goals ┆ prop_total_goals │
│ ---       ┆ ---             ┆ ---               ┆ --- ┆ ---         ┆ ---              │
│ str       ┆ str             ┆ str               ┆ i16 ┆ i16         ┆ f32              │
╞═══════════╪═════════════════╪═══════════════════╪═════╪═════════════╪══════════════════╡
│ 1994-1995 ┆ Blackburn       ┆ Alan Shearer      ┆ 34  ┆ 78          ┆ 0.435897         │
│ 2003-2004 ┆ Arsenal         ┆ Thierry Henry     ┆ 30  ┆ 69          ┆ 0.434783         │
│ 2007-2008 ┆ Manchester Utd  ┆ Cristiano Ronaldo ┆ 31  ┆ 78          ┆ 0.397436         │
│ 2022-2023 ┆ Manchester City ┆ Erling Haaland    ┆ 36  ┆ 92          ┆ 0.391304         │
│ 2015-2016 ┆ Leicester City  ┆ Jamie Vardy       ┆ 24  ┆ 68          ┆ 0.352941         │
│ …         ┆ …               ┆ …                 ┆ …   ┆ …           ┆ …  

# Entire Scorer Distribution
First, need to get a data structure of `season: champion's goal distribution` which will be probably a `dict[str: array]`.

There certainly won't be the same amount of goal scorers per season, so there are three "standardization" approaches that come to mind:
  1. If there's always been a 25-man roster, then get the 25 names and there will just be a lot of mass on 0
  
   - Moreover, this is unreasonable because goalkeepers _almost never score_

  2. Very similar to (1) above: get the number of players who made an appearance and use _that_ as the support of the goals scored dist'n

   - Same problem with the goalkeepers never scoring, "artifically" inflating the support

  3. Subset the champions' squads to _just_ those who scored 1 goal: the support of the dist'n will be that number of players

   - This _feels_ more like the basis of a comparison-to-uniform; i.e., for those who pitched in: how far is the actual dist'n from an "egalitarian" goal spread?

Then, if it looks wonky for some reason, could do a further analysis that limits to _just_ the players that scored more than once (this would necessitate subtracting from the total goals) because those could be flukes, and see what the dist'n for those $P$ players is.

In [ ]:
# Start with Liverpool 2024-2025


In [19]:
goals25 = subset25.with_columns(
    prop_total_goals=pl.col("Gls").truediv(pl.col("Gls").sum())
)
goals25

Squad,Player,Age,Min,Gls,Ast,prop_total_goals
str,str,i16,i16,i16,i16,f64
"""Liverpool""","""Mohamed Salah""",32,3371,29,18,0.341176
"""Liverpool""","""Luis Díaz""",27,2399,13,5,0.152941
"""Liverpool""","""Cody Gakpo""",25,1935,10,4,0.117647
"""Liverpool""","""Diogo Jota""",27,1196,6,3,0.070588
"""Liverpool""","""Dominik Szoboszlai""",23,2491,6,6,0.070588
…,…,…,…,…,…,…
"""Liverpool""","""Vitezslav Jaros""",23,12,0,0,0.0
"""Liverpool""","""Caoimhín Kelleher""",25,900,0,0,0.0
"""Liverpool""","""Jarell Quansah""",21,495,0,0,0.0


In [19]:
len(goals25)

24

In [15]:
from scipy.stats import energy_distance, wasserstein_distance

In [20]:
wasserstein_distance(
    [1 / goals25.shape[0]] * goals25.shape[0],
    goals25.select("prop_total_goals").to_series(),
)

np.float64(0.04824346405228758)

In [22]:
s = goals25.select("prop_total_goals").to_series()
s.filter(s > 0)

prop_total_goals
f64
0.341176
0.152941
0.117647
0.070588
0.070588
…
0.035294
0.035294
0.035294


In [23]:
wasserstein_distance(
    pl.Series((1/len(s.filter(s > 0)) for x in s.filter(s > 0))),
    s.filter(s > 0)
)

np.float64(0.06029411764705883)

In [24]:
wasserstein_distance(
    pl.Series([1.0] + [0.0]*(len(s.filter(s>0)) - 1)),
    s.filter(s > 0)
)

np.float64(0.10980392156862746)

In [25]:
pl.Series([1.0] + [0.0]*(len(s.filter(s>0)) - 1))

""
f64
1.0
0.0
0.0
0.0
0.0
…
0.0
0.0
0.0


In [21]:
def uniform_n(s: pl.Series) -> pl.Series:
    """Return a ``pl.Series`` populated by the scalar ``1 / len(s)``."""
    # a.k.a. flat Dirichlet distribution
    # https://en.wikipedia.org/wiki/Dirichlet_distribution#Special_cases
    return pl.Series(1/len(s) for _ in s)

In [27]:
energy_distance(
    [1 / len(goals25)] * len(goals25),
    goals25.select("prop_total_goals").to_series(),
)

np.float64(0.1866675420147543)

In [22]:
all_squads: list[pl.DataFrame, ...] = [None] * len(CHAMPIONS)

In [28]:
pldf = pl.from_pandas(html_to_flat_dataframe(Path(f"data/premier-league/fbref_{season}.html")))
pldf

Squad,Player,Age,Min,Gls,Ast
str,str,i16,i16,i16,i16
"""Everton""","""Gary Ablett""",26,3600,0,2
"""Southampton""","""Micky Adams""",30,3245,4,3
"""Oldham Athletic""","""Neil Adams""",26,2454,8,2
"""Arsenal""","""Tony Adams""",25,3005,0,0
"""Southampton""","""Derek Allan""",17,11,0,0
…,…,…,…,…,…
"""Middlesbrough""","""Tommy Wright""",26,3044,5,8
"""Ipswich Town""","""Frank Yallop""",28,452,2,0
"""Aston Villa""","""Dwight Yorke""",20,1918,6,3


# All Seasons, Yo

In [66]:
l = [None] * len(CHAMPIONS)

for i, (season, champ) in enumerate(CHAMPIONS.items()):
    pdf = html_to_flat_dataframe(Path(f"data/premier-league/fbref_{season}.html"))
    pldf = pl.from_pandas(pdf)

    l[i] = pldf.with_columns(
        pl.col("Gls").sum().over(pl.col("Squad")).alias("total_goals").cast(pl.Int16),
        pl.col("Gls")
        .truediv(pl.col("Gls").sum().over(pl.col("Squad")))
        .alias("prop_total_goals")
        .cast(pl.Float32),
        season=pl.lit(season),
    )

    # l[i] = pldf.group_by("Squad").agg(
    #     pl.col("Gls").truediv(pl.col("Gls").sum()).max().alias("max_prop_goals"),
    # ).with_columns(season=pl.lit(season))

foo = pl.concat(l)


# with_columns(
#     pl.col("Gls").truediv(pl.col("Gls").sum()).alias("prop_total_goals").cast(pl.Float32),
# ).sort(by="Gls", descending=True).slice(0, 1)

In [71]:
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(foo.sort(by="prop_total_goals", descending=True).select("season", "Squad", "Player", "Gls", "total_goals", "prop_total_goals").head(n=50))

shape: (50, 6)
┌───────────┬─────────────────┬───────────────────────────┬─────┬─────────────┬──────────────────┐
│ season    ┆ Squad           ┆ Player                    ┆ Gls ┆ total_goals ┆ prop_total_goals │
│ ---       ┆ ---             ┆ ---                       ┆ --- ┆ ---         ┆ ---              │
│ str       ┆ str             ┆ str                       ┆ i16 ┆ i16         ┆ f32              │
╞═══════════╪═════════════════╪═══════════════════════════╪═════╪═════════════╪══════════════════╡
│ 2002-2003 ┆ Southampton     ┆ James Beattie             ┆ 23  ┆ 42          ┆ 0.547619         │
│ 2016-2017 ┆ Sunderland      ┆ Jermain Defoe             ┆ 15  ┆ 28          ┆ 0.535714         │
│ 2009-2010 ┆ Sunderland      ┆ Darren Bent               ┆ 24  ┆ 45          ┆ 0.533333         │
│ 1999-2000 ┆ Sunderland      ┆ Kevin Phillips            ┆ 30  ┆ 57          ┆ 0.526316         │
│ 2021-2022 ┆ Norwich City    ┆ Teemu Pukki               ┆ 11  ┆ 21          ┆ 0.52381       

In [73]:
foo.sort(by="total_goals", descending=True)

Squad,Player,Age,Min,Gls,Ast,total_goals,prop_total_goals,season
str,str,i16,i16,i16,i16,i16,f32,str
"""Manchester City""","""Sergio Agüero""",29,1963,21,6,103,0.203883,"""2017-2018"""
"""Manchester City""","""Claudio Bravo""",34,226,0,0,103,0.0,"""2017-2018"""
"""Manchester City""","""Danilo""",26,1349,3,2,103,0.029126,"""2017-2018"""
"""Manchester City""","""Kevin De Bruyne""",26,3073,8,16,103,0.07767,"""2017-2018"""
"""Manchester City""","""Fabian Delph""",27,1741,1,2,103,0.009709,"""2017-2018"""
…,…,…,…,…,…,…,…,…
"""Derby County""","""Mile Sterjovski""",28,688,0,1,19,0.0,"""2007-2008"""
"""Derby County""","""Alan Stubbs""",35,645,0,0,19,0.0,"""2007-2008"""
"""Derby County""","""Gary Teale""",29,917,0,1,19,0.0,"""2007-2008"""


In [23]:
for i, (season, champ) in enumerate(CHAMPIONS.items()):
    pdf = html_to_flat_dataframe(Path(f"data/premier-league/fbref_{season}.html"))
    pldf = pl.from_pandas(pdf).filter(pl.col("Squad").eq(champ))

    goal_distribution: pl.DataFrame = pldf.with_columns(
        pl.lit(season).alias("Season"),
        pl.lit(champ).alias("Champion"),
        pl.col("Gls").sum().alias("squad_goals").cast(pl.Int16),
        pl.col("Gls").truediv(pl.col("Gls").sum()).alias("prop_squad_goals"),
    ).sort(by="prop_squad_goals", descending=True)

    distn_squad: pl.Series = goal_distribution.select("prop_squad_goals").to_series()

    distn_scorers: pl.Series = (
        goal_distribution.filter(pl.col("Gls").gt(0))
        .select("prop_squad_goals")
        .to_series()
    )
    wd_squad = wasserstein_distance(uniform_n(distn_squad), distn_squad)
    wd_scorers = wasserstein_distance(uniform_n(distn_scorers), distn_scorers)
    ed_squad = energy_distance(uniform_n(distn_squad), distn_squad)
    ed_scorers = energy_distance(uniform_n(distn_scorers), distn_scorers)

    distn_dist_df = goal_distribution.with_columns(
        max_prop_squad_goals=pl.lit(distn_scorers.max()),
        wass_dist_squad=pl.lit(wd_squad),
        wass_dist_scorers=pl.lit(wd_scorers),
        ener_dist_squad=pl.lit(ed_squad),
        ener_dist_scorers=pl.lit(ed_scorers),
    )
    print(distn_dist_df)

    all_squads[i] = distn_dist_df.select(
        "Season",
        "Champion",
        "squad_goals",
        "max_prop_squad_goals",
        "wass_dist_squad",
        "wass_dist_scorers",
        "ener_dist_squad",
        "ener_dist_scorers",
    ).slice(0, 1)

shape: (24, 15)
┌───────────┬─────────────┬─────┬──────┬───┬─────────────┬─────────────┬─────────────┬─────────────┐
│ Squad     ┆ Player      ┆ Age ┆ Min  ┆ … ┆ wass_dist_s ┆ wass_dist_s ┆ ener_dist_s ┆ ener_dist_s │
│ ---       ┆ ---         ┆ --- ┆ ---  ┆   ┆ quad        ┆ corers      ┆ quad        ┆ corers      │
│ str       ┆ str         ┆ i16 ┆ i16  ┆   ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│           ┆             ┆     ┆      ┆   ┆ f64         ┆ f64         ┆ f64         ┆ f64         │
╞═══════════╪═════════════╪═════╪══════╪═══╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Liverpool ┆ Mohamed     ┆ 32  ┆ 3371 ┆ … ┆ 0.048243    ┆ 0.060294    ┆ 0.186668    ┆ 0.201708    │
│           ┆ Salah       ┆     ┆      ┆   ┆             ┆             ┆             ┆             │
│ Liverpool ┆ Luis Díaz   ┆ 27  ┆ 2399 ┆ … ┆ 0.048243    ┆ 0.060294    ┆ 0.186668    ┆ 0.201708    │
│ Liverpool ┆ Cody Gakpo  ┆ 25  ┆ 1935 ┆ … ┆ 0.048243    ┆ 0.060294    ┆ 0.

In [ ]:
pl.concat(all_squads).with_columns(
    pl.col("wass_dist_squad")
    .rank(method="dense", descending=True)
    .alias("rank_wass_dist_squad"),
    pl.col("wass_dist_scorers")
    .rank(method="dense", descending=True)
    .alias("rank_wass_dist_scorers"),
    pl.col("max_prop_squad_goals")
    .rank(method="dense", descending=True)
    .alias("rank_max_prop_squad_goals"),
).select(
    "Season",
    "Champion",
    "squad_goals",
    "max_prop_squad_goals",
    "rank_max_prop_squad_goals",
    "wass_dist_scorers",
    "rank_wass_dist_scorers",
).sort(by=pl.col("rank_wass_dist_scorers"))

Season,Champion,squad_goals,max_prop_squad_goals,rank_max_prop_squad_goals,wass_dist_scorers,rank_wass_dist_scorers
str,str,i16,f64,u32,f64,u32
"""1994-1995""","""Blackburn""",78,0.435897,1,0.081161,1
"""2003-2004""","""Arsenal""",69,0.434783,2,0.078502,2
"""2007-2008""","""Manchester Utd""",78,0.397436,3,0.076923,3
"""2002-2003""","""Manchester Utd""",73,0.342466,6,0.072045,4
"""2015-2016""","""Leicester City""",68,0.352941,5,0.070832,5
"""2009-2010""","""Chelsea""",99,0.292929,10,0.065966,6
"""2001-2002""","""Arsenal""",79,0.303797,9,0.065463,7
"""2022-2023""","""Manchester City""",92,0.391304,4,0.060386,8
"""2024-2025""","""Liverpool""",85,0.341176,7,0.060294,9


\# Dirichlet and Statistical Distance
There is at least one [academic paper](https://www.sciencedirect.com/science/article/abs/pii/S0031320307003123) discussing the distance between Dirichlet distribution realizations, and a [smart-looking blog post](https://benmoran.wordpress.com/2012/07/11/distances-divergences-dirichlet-distributions/).

The form of the Dirichlet distribution imposes that $\alpha_k > 0,\, \, \, k=1,...,K$ and that $||\vec{\alpha}||_1 = 1,\,\,\, \vec{\alpha}\in\mathbb{R}^K$. This means that
$$\mathbb{R}^{K}_{\geq 0,\,\leq 1}\ni\vec{G}_{cs}\sim \text{Dir}(\vec{\alpha}, K)$$

Based on the above paper, only the Chernoff distance (which simplifies to the Bhattacharyya distance when $\lambda = \frac{1}{2}$) is appropriate as a distance metric between Dirichlet distributions. So, instead of using the energy distance and ~~Wasserstein~~ Kantorovich distance (see [here](https://en.wikipedia.org/wiki/Wasserstein_metric#)), we shall use the Bhattacharyya distance instead. **N.b.** this may be overkill/inappropriate/unnecessary because of the comparison to the "flat Dirichlet", a.k.a. $K$-dimension uniform.

## Chernoff Distance
$$J_C(\vec{\alpha}_a,\vec{\alpha}_b) = \ln\Gamma\bigg(\sum_{k=1}^{K}\lambda\alpha_{ak} + (1-\lambda)\alpha_{bk}\bigg) + \lambda\sum_{k=1}^{K}\ln\Gamma(\alpha_{ak}) + (1-\lambda)\sum_{k=1}^{K}\ln\Gamma(\alpha_{bk})$$

In [ ]:
s = goals25.select("prop_total_goals").to_series()
s

prop_total_goals
f64
0.341176
0.152941
0.117647
0.070588
0.070588
…
0.0
0.0
0.0


In [62]:
import numpy as np
from scipy.spatial import distance
from scipy.special import gamma, gammaln

assert distance.euclidean(s, s) == 0.0

In [38]:
# Is this the only multivariate distance metric in scipy?
distance.mahalanobis(s, s)

TypeError: mahalanobis() missing 1 required positional argument: 'VI'

In [42]:
s

prop_total_goals
f64
0.341176
0.152941
0.117647
0.070588
0.070588
…
0.0
0.0
0.0


## Distance to All-Mass-On-One-Component Vector
What is the distance between the observed $\mathbb{R}^{K}_{\geq 0,\,\leq 1}\ni\vec{G}^{*}$ and the opposite of a uniform (a vniform??) in the same space that is entirely "peaked" at one component?

Also, are these distances the same independent of which $k=1,...,K$ has all the mass?

In [50]:
# a "vniform" with all mass on the 0th element
v: np.ndarray = np.eye(len(s))
v0: pl.Series = pl.Series(v[0])
v1: pl.Series = pl.Series(v[1])
# etc.

In [58]:
np.cov([s, v0], rowvar=False).shape

(24, 24)

In [ ]:
distance.mahalanobis(
    s,
    v0,
    np.linalg.inv(np.cov([s, v0], rowvar=False)),
)

LinAlgError: Singular matrix

Okay, so whatever I'm doing is giving me covariance matrices that are not copacetic...

In [63]:
distance.euclidean(s, v0)

0.7015553654529681

In [64]:
distance.euclidean(s, v1)

0.9320142268394522

In [65]:
distance.euclidean(s, pl.Series(v[2]))

0.9691433094879516

Move on to roll my own Bhattacharyya distance.

In [66]:
def bhattacharyya(
    p: np.typing.ArrayLike,
    q: np.typing.ArrayLike,
) -> float:
    lmbda: float = 0.5
    return lmbda